In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Setup and Imports

In [31]:
import pandas as pd
import numpy as np
from numpy.linalg import norm


In [3]:
OHCO = ['doc_title', 'para_num', 'sentence_num', 'token_num']
bags = dict(
    SENTS = OHCO[:3],
    PARAS = OHCO[:2],
    STORIES = OHCO[:1]
)
bag = 'STORIES'


In [4]:
OHCO[:1]

['doc_title']

In [5]:
VOCAB = pd.read_csv('data/pg2591-VOCAB.csv')
VOCAB.set_index('term_str',inplace=True)
VOCAB.head()

,n,p,i,n_chars,max_pos_group,max_pos,p_stem,stop
term_str,,,,,,,,
the,6927,0.068553,3.866638,3,DT,DT,the,1
and,5433,0.053768,4.217119,3,CC,CC,and,1
to,2662,0.026344,5.246358,2,TO,TO,to,1
he,2093,0.020713,5.593296,2,PR,PRP,he,1
a,1909,0.018892,5.726051,1,DT,DT,a,1


In [6]:
TOKENS = pd.read_csv("data/p2591-TOKENS.csv").set_index(OHCO).dropna()
TOKENS=TOKENS.reset_index()
TOKENS.head()

,doc_title,para_num,sentence_num,token_num,pos_tuple,pos,token_str,term_str,pos_group
0,ASHPUTTEL,0,0,0,"('The', 'DT')",DT,The,the,DT
1,ASHPUTTEL,0,0,1,"('wife', 'NN')",NN,wife,wife,NN
2,ASHPUTTEL,0,0,2,"('of', 'IN')",IN,of,of,IN
3,ASHPUTTEL,0,0,3,"('a', 'DT')",DT,a,a,DT
4,ASHPUTTEL,0,0,4,"('rich', 'JJ')",JJ,rich,rich,JJ


## BOW by STORIES

In [7]:
BOW = TOKENS.groupby(bags[bag]+['term_str']).term_str.count().to_frame('n') 
BOW

n
doc_title term_str      
ASHPUTTEL _my_         1
          a           21
          about        3
          after        2
          afterwards   1
...                   ..
TOM THUMB would        5
          yes          1
          yet          4
          you         35
          your         5

[26654 rows x 1 columns]

## DTCM Table

In [8]:
DTCM = BOW.n.unstack(fill_value=0)
DTCM.head(10)

term_str,1,2,3,_jug,_my_,a,abashed,abide,able,abode,...,yonder,you,young,younger,youngest,your,yours,yourself,yourselves,youth
doc_title,,,,,,,,,,,,,,,,,,,,,
ASHPUTTEL,0,0,0,0,1,21,0,0,0,0,...,0,24,0,0,0,3,0,0,0,0
BRIAR ROSE,0,0,0,0,0,33,0,0,0,0,...,0,2,1,0,0,2,0,0,0,0
CAT AND MOUSE IN PARTNERSHIP,0,0,0,0,0,17,0,0,0,0,...,0,21,0,0,0,4,1,1,0,0
CAT-SKIN,0,0,0,0,0,46,0,0,0,0,...,0,20,0,0,0,1,0,0,0,0
CLEVER ELSIE,0,0,0,0,0,23,0,0,0,0,...,0,4,0,0,0,0,0,0,0,0
CLEVER GRETEL,0,0,0,0,0,14,0,0,0,0,...,0,9,0,0,0,1,0,1,0,0
CLEVER HANS,0,0,0,0,0,15,0,0,0,0,...,0,33,1,0,0,3,0,0,0,0
DOCTOR KNOWALL,0,0,0,0,0,20,0,0,0,0,...,0,2,0,0,0,3,0,3,0,0
FREDERICK AND CATHERINE,0,0,0,0,0,30,0,0,0,0,...,0,32,0,1,0,2,0,0,0,0


In [9]:
print(TOKENS.head())
print(TOKENS.index.names)
bags[bag]

   doc_title  para_num  sentence_num  token_num       pos_tuple pos token_str  \
0  ASHPUTTEL         0             0          0   ('The', 'DT')  DT       The   
1  ASHPUTTEL         0             0          1  ('wife', 'NN')  NN      wife   
2  ASHPUTTEL         0             0          2    ('of', 'IN')  IN        of   
3  ASHPUTTEL         0             0          3     ('a', 'DT')  DT         a   
4  ASHPUTTEL         0             0          4  ('rich', 'JJ')  JJ      rich   

  term_str pos_group  
0      the        DT  
1     wife        NN  
2       of        IN  
3        a        DT  
4     rich        JJ  
[None]


['doc_title']

## Making TFIDF

In [10]:
tf_method = 'sum'         # sum, max, log, double_norm, raw, binary
tf_norm_k = .5            # only used for double_norm
idf_method = 'standard'   # standard, max, smooth
gradient_cmap = 'YlGnBu'  # YlGn, GnBu, YlGnBu; For tables; see https://matplotlib.org/3.1.0/tutorials/colors/colormaps.html 

In [11]:
print('TF method:', tf_method)

if tf_method == 'sum':
    TF = DTCM.T / DTCM.T.sum()

elif tf_method == 'max':
    TF = DTCM.T / DTCM.T.max()
    
elif tf_method == 'log':
    TF = np.log2(1 + DTCM.T)
    
elif tf_method == 'raw':
    TF = DTCM.T
    
elif tf_method == 'double_norm':
    TF = DTCM.T / DTCM.T.max()
    
elif tf_method == 'binary':
    TF = DTCM.T.astype('bool').astype('int')
    
TF = TF.T

TF method: sum


In [12]:
TF.head(10)

term_str,1,2,3,_jug,_my_,a,abashed,abide,able,abode,...,yonder,you,young,younger,youngest,your,yours,yourself,yourselves,youth
doc_title,,,,,,,,,,,,,,,,,,,,,
ASHPUTTEL,0.0,0.0,0.0,0.0,0.000386,0.008111,0.0,0.0,0.0,0.0,...,0.0,0.009270,0.000000,0.000000,0.0,0.001159,0.000000,0.000000,0.0,0.0
BRIAR ROSE,0.0,0.0,0.0,0.0,0.000000,0.021739,0.0,0.0,0.0,0.0,...,0.0,0.001318,0.000659,0.000000,0.0,0.001318,0.000000,0.000000,0.0,0.0
CAT AND MOUSE IN PARTNERSHIP,0.0,0.0,0.0,0.0,0.000000,0.017259,0.0,0.0,0.0,0.0,...,0.0,0.021320,0.000000,0.000000,0.0,0.004061,0.001015,0.001015,0.0,0.0
CAT-SKIN,0.0,0.0,0.0,0.0,0.000000,0.021536,0.0,0.0,0.0,0.0,...,0.0,0.009363,0.000000,0.000000,0.0,0.000468,0.000000,0.000000,0.0,0.0
CLEVER ELSIE,0.0,0.0,0.0,0.0,0.000000,0.017190,0.0,0.0,0.0,0.0,...,0.0,0.002990,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0
CLEVER GRETEL,0.0,0.0,0.0,0.0,0.000000,0.014973,0.0,0.0,0.0,0.0,...,0.0,0.009626,0.000000,0.000000,0.0,0.001070,0.000000,0.001070,0.0,0.0
CLEVER HANS,0.0,0.0,0.0,0.0,0.000000,0.016556,0.0,0.0,0.0,0.0,...,0.0,0.036424,0.001104,0.000000,0.0,0.003311,0.000000,0.000000,0.0,0.0
DOCTOR KNOWALL,0.0,0.0,0.0,0.0,0.000000,0.025674,0.0,0.0,0.0,0.0,...,0.0,0.002567,0.000000,0.000000,0.0,0.003851,0.000000,0.003851,0.0,0.0
FREDERICK AND CATHERINE,0.0,0.0,0.0,0.0,0.000000,0.015649,0.0,0.0,0.0,0.0,...,0.0,0.016693,0.000000,0.000522,0.0,0.001043,0.000000,0.000000,0.0,0.0


In [13]:
DTCM.astype(bool)

term_str,1,2,3,_jug,_my_,a,abashed,abide,able,abode,...,yonder,you,young,younger,youngest,your,yours,yourself,yourselves,youth
doc_title,,,,,,,,,,,,,,,,,,,,,
ASHPUTTEL,False,False,False,False,True,True,False,False,False,False,...,False,True,False,False,False,True,False,False,False,False
BRIAR ROSE,False,False,False,False,False,True,False,False,False,False,...,False,True,True,False,False,True,False,False,False,False
CAT AND MOUSE IN PARTNERSHIP,False,False,False,False,False,True,False,False,False,False,...,False,True,False,False,False,True,True,True,False,False
CAT-SKIN,False,False,False,False,False,True,False,False,False,False,...,False,True,False,False,False,True,False,False,False,False
CLEVER ELSIE,False,False,False,False,False,True,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
THE WEDDING OF MRS. FOX - PART II,False,False,False,False,False,True,False,False,False,False,...,False,True,True,False,False,False,False,False,False,False
THE WHITE SNAKE,False,False,False,False,False,True,False,False,False,False,...,False,True,True,False,False,False,False,False,True,True
THE WILLOW-WREN AND THE BEAR,False,False,False,False,False,True,False,False,False,False,...,False,True,True,False,False,True,False,False,False,False


In [14]:
DF = DTCM.astype('bool').sum() 
DF

term_str
1              1
2              1
3              1
_jug           1
_my_           1
              ..
your          51
yours          3
yourself      23
yourselves     4
youth          4
Length: 4804, dtype: int64

In [15]:
N = DTCM.shape[0]

N

62

In [16]:
print('IDF method:', idf_method)

if idf_method == 'standard':
    IDF = np.log2(N / DF)

elif idf_method == 'max':
    IDF = np.log2(DF.max() / DF) 

elif idf_method == 'plus':
    IDF = np.log2(N / DF) + 1

# This is what SciKit Learn uses
elif idf_method == 'smooth':
    IDF = np.log2((1 + N) / (1 + DF)) + 1

IDF method: standard


In [17]:
TFIDF = TF * IDF
TFIDF.head()

term_str,1,2,3,_jug,_my_,a,abashed,abide,able,abode,...,yonder,you,young,younger,youngest,your,yours,yourself,yourselves,youth
doc_title,,,,,,,,,,,,,,,,,,,,,
ASHPUTTEL,0.0,0.0,0.0,0.0,0.0023,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000439,0.000000,0.0,0.0,0.000327,0.000000,0.000000,0.0,0.0
BRIAR ROSE,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000062,0.000755,0.0,0.0,0.000371,0.000000,0.000000,0.0,0.0
CAT AND MOUSE IN PARTNERSHIP,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.001009,0.000000,0.0,0.0,0.001144,0.004436,0.001452,0.0,0.0
CAT-SKIN,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000443,0.000000,0.0,0.0,0.000132,0.000000,0.000000,0.0,0.0
CLEVER ELSIE,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000141,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0


In [18]:
DF.index

Index(['1', '2', '3', '_jug', '_my_', 'a', 'abashed', 'abide', 'able', 'abode',
       ...
       'yonder', 'you', 'young', 'younger', 'youngest', 'your', 'yours',
       'yourself', 'yourselves', 'youth'],
      dtype='object', name='term_str', length=4804)

In [19]:
VOCAB.index

Index(['the', 'and', 'to', 'he', 'a', 'was', 'of', 'it', 'she', 'you',
       ...
       'clap', 'mount', 'removed', 'fattening', 'stops', 'bath',
       'bathinghouse', 'arrive', 'impatience', 'fears'],
      dtype='object', name='term_str', length=4804)

In [20]:
vocab_set = set(VOCAB.index)
df_set = set(DF.index)

extra_in_vocab = vocab_set - df_set
extra_in_df = df_set - vocab_set

print("Extra in vocab:", extra_in_vocab)
print("Extra in DF:", extra_in_df)

Extra in vocab: set()
Extra in DF: set()


In [21]:
print(VOCAB.index.equals(DF.index))
print(VOCAB.index.equals(IDF.index))
print(VOCAB.index.equals(TFIDF.mean().index))

False
False
False


In [22]:
VOCAB['df'] = DF
VOCAB['idf'] = IDF
VOCAB['tfidf_mean'] = TFIDF.mean()

## Making Normalized Reduced TFIDF

In [23]:
N = TOKENS.reset_index().value_counts(['doc_title']).shape[0]

VOCAB['tfidf_mean'] = TFIDF.mean() 
VOCAB['dfidf'] = VOCAB.df * VOCAB.idf # DFIDF is the same as mean boolean TFIDF
VOCAB['dp'] = VOCAB.df / N
VOCAB['di'] = np.log2(1/VOCAB.dp)
VOCAB['dh'] = VOCAB.dp * VOCAB.di
VOCAB

,n,p,i,n_chars,max_pos_group,max_pos,p_stem,stop,df,idf,tfidf_mean,dfidf,dp,di,dh
term_str,,,,,,,,,,,,,,,
the,6927,0.068553,3.866638,3,DT,DT,the,1,62,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
and,5433,0.053768,4.217119,3,CC,CC,and,1,62,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
to,2662,0.026344,5.246358,2,TO,TO,to,1,62,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
he,2093,0.020713,5.593296,2,PR,PRP,he,1,62,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
a,1909,0.018892,5.726051,1,DT,DT,a,1,62,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
bath,1,0.000010,16.624653,4,NN,NN,bath,0,1,5.954196,0.000038,5.954196,0.016129,5.954196,0.096035
bathinghouse,1,0.000010,16.624653,12,NN,NN,bathinghous,0,1,5.954196,0.000038,5.954196,0.016129,5.954196,0.096035
arrive,1,0.000010,16.624653,6,VB,VB,arriv,0,1,5.954196,0.000038,5.954196,0.016129,5.954196,0.096035


In [27]:
thresh = VOCAB.dh.quantile(.8).round(3) # Lowered threshold because i was getting such little results
float(thresh)

0.293

In [28]:
SIGS = VOCAB[VOCAB.dh >= thresh]
SIGS[['max_pos', 'n', 'i', 'dh']].sort_values('dh', ascending=False)\
    .head(25).style.background_gradient(cmap='YlGnBu')

,max_pos,n,i,dh
term_str,,,,
standing,VBG,40,11.302725,0.530719
yourself,PRP,31,11.670456,0.530719
threw,VBD,43,11.198388,0.530719
most,RBS,31,11.670456,0.530719
forced,VBN,30,11.717762,0.530719
even,RB,32,11.624653,0.530719
grew,VBD,34,11.537190,0.530719
ready,JJ,37,11.415199,0.530719
drew,VBD,36,11.454728,0.530400


In [29]:
TFIDF_SIGS = TFIDF[SIGS.index].copy()
TFIDF_SIGS

term_str,king,man,old,we,upon,father,himself,shall,take,cried,...,appear,cooking,grass,backwards,finding,doors,sides,searched,started,hide
doc_title,,,,,,,,,,,,,,,,,,,,,
ASHPUTTEL,0.005081,0.000217,0.000191,0.000191,0.000665,0.004658,0.000536,0.000665,0.000463,0.000998,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.001301
BRIAR ROSE,0.010832,0.000740,0.002607,0.000000,0.001702,0.000000,0.000000,0.000851,0.000526,0.000284,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
CAT AND MOUSE IN PARTNERSHIP,0.000000,0.000000,0.000000,0.003516,0.000437,0.000000,0.000000,0.000874,0.000406,0.001312,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
CAT-SKIN,0.019502,0.000000,0.000000,0.000927,0.000605,0.001540,0.000000,0.000202,0.001122,0.000000,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.001577
CLEVER ELSIE,0.000000,0.002100,0.000000,0.004437,0.000000,0.001639,0.000000,0.001931,0.000000,0.000644,...,0.002518,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
THE WEDDING OF MRS. FOX - PART II,0.000000,0.001494,0.005263,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
THE WHITE SNAKE,0.007476,0.001045,0.000307,0.003374,0.001068,0.000000,0.001147,0.000267,0.000000,0.001068,...,0.000000,0.0,0.002089,0.0,0.002089,0.0,0.000000,0.000000,0.000000,0.000000
THE WILLOW-WREN AND THE BEAR,0.007173,0.000000,0.001079,0.005935,0.000470,0.000000,0.000000,0.002348,0.000872,0.001409,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.003674,0.000000,0.003674,0.000000


### Normalizing Reduced Table

In [32]:
L2 = TFIDF_SIGS.apply(lambda x: x / norm(x), 1) # Pythagorean, AKA Euclidean
L2


term_str,king,man,old,we,upon,father,himself,shall,take,cried,...,appear,cooking,grass,backwards,finding,doors,sides,searched,started,hide
doc_title,,,,,,,,,,,,,,,,,,,,,
ASHPUTTEL,0.111674,0.004770,0.004200,0.004200,0.014623,0.102368,0.011775,0.014623,0.010177,0.021935,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.028603
BRIAR ROSE,0.263257,0.017991,0.063370,0.000000,0.041367,0.000000,0.000000,0.020683,0.012795,0.006894,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
CAT AND MOUSE IN PARTNERSHIP,0.000000,0.000000,0.000000,0.038154,0.004744,0.000000,0.000000,0.009488,0.004402,0.014232,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
CAT-SKIN,0.452256,0.000000,0.000000,0.021486,0.014026,0.035704,0.000000,0.004675,0.026031,0.000000,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.036579
CLEVER ELSIE,0.000000,0.038482,0.000000,0.081325,0.000000,0.030031,0.000000,0.035392,0.000000,0.011797,...,0.046151,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
THE WEDDING OF MRS. FOX - PART II,0.000000,0.011494,0.040485,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
THE WHITE SNAKE,0.200396,0.028013,0.008222,0.090446,0.028627,0.000000,0.030734,0.007157,0.000000,0.028627,...,0.000000,0.0,0.055993,0.0,0.055993,0.0,0.000000,0.000000,0.000000,0.000000
THE WILLOW-WREN AND THE BEAR,0.098323,0.000000,0.014792,0.081358,0.006438,0.000000,0.000000,0.032188,0.011947,0.019313,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.050366,0.000000,0.050366,0.000000


## Save Reduced and Normlaizd TFIDF table

In [33]:
output_dir ="data"

In [34]:
L2.to_csv(f"{output_dir}/pg2591-TFIDF.csv", index=True)